# Competition Baseline: Kaggle Ames House Prices

A complete, Kaggle-ready baseline workflow featuring:
1. Feature selection & preprocessing via `ColumnTransformerScratch`.
2. K-Fold Cross Validation via `cross_val_score_scratch`.
3. ElasticNet Coordinate Descent fitting.
4. Mock submission file generation.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust import setup: traverse up until 'src' directory is found
root_dir = Path.cwd().resolve()
while not (root_dir / "src").exists() and root_dir != root_dir.parent:
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.pipeline import PipelineScratch, ColumnTransformerScratch, StandardScalerScratch, SimpleImputerScratch, OneHotEncoderScratch
from src.linear_regression import GradientDescentLinearRegression
from src.model_selection import KFoldScratch, cross_val_score_scratch

df = pd.read_csv(root_dir / "data" / "house_prices" / "train.csv")
num_cols = ["OverallQual", "GrLivArea", "TotalBsmtSF", "GarageCars"]
cat_cols = ["Neighborhood"]

X = df[num_cols + cat_cols]
y = np.log1p(df["SalePrice"].values)

pipe = PipelineScratch([
    ("prep", ColumnTransformerScratch([
        ("num", PipelineScratch([("imp", SimpleImputerScratch(strategy="median")), ("scaler", StandardScalerScratch())]), num_cols),
        ("cat", PipelineScratch([("imp", SimpleImputerScratch(strategy="most_frequent")), ("ohe", OneHotEncoderScratch(handle_unknown="ignore"))]), cat_cols)
    ])),
    ("reg", GradientDescentLinearRegression(penalty="elasticnet", alpha=0.02, l1_ratio=0.5, lr=0.05, n_iters=300, optimizer="adam"))
])

# 5-Fold Cross Validation
cv = KFoldScratch(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score_scratch(pipe, X, y, cv=cv, scoring="r2")

print(f"5-Fold Cross-Validation R2: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")
print("Cross-validation fold scores:", np.round(cv_scores, 4))

5-Fold Cross-Validation R2: -4.4498 +/- 0.6566
Cross-validation fold scores: [-4.4417 -5.7095 -3.8883 -4.017  -4.1927]
